# Backstage: Minimale GitHub-Integration

Dieses Notebook beschreibt die Schritte **1 bis 4** für die einfachste GitHub-Integration ohne Kubernetes.

## Ziel

Backstage soll alle Repositories einer GitHub-Organisation durchsuchen und diejenigen automatisch in den Software Catalog aufnehmen, die im Hauptverzeichnis eine Datei namens `catalog-info.yaml` enthalten.

> Der GitHub Discovery Provider arbeitet mit einer **GitHub-Organisation** oder einer GitHub App. Für diese minimale Variante wird eine Organisation und ein Personal Access Token verwendet.

**Offizielle Dokumentation**

- [GitHub Locations](https://backstage.io/docs/integrations/github/locations/)
- [GitHub Discovery](https://backstage.io/docs/integrations/github/discovery/)


## 1. GitHub Personal Access Token erstellen

Erstelle in GitHub einen Personal Access Token.

Für das Lesen von Software-Komponenten benötigt der klassische Token den Scope:

```text
repo
```

Der Token wird nicht direkt in `app-config.yaml` eingetragen, sondern über die Umgebungsvariable `GITHUB_TOKEN` bereitgestellt.

Ersetze im nächsten Befehl den Beispielwert durch deinen Token.


In [ ]:
# Nur für die aktuelle Notebook-/Python-Session.
# Für den Start von Backstage muss die Variable auch in der Shell gesetzt sein.

import os

os.environ["GITHUB_TOKEN"] = "github_pat_HIER_TOKEN_EINTRAGEN"

print("GITHUB_TOKEN wurde für diese Session gesetzt.")


In einer Linux-Shell setzt du den Token so:

```bash
export GITHUB_TOKEN='github_pat_HIER_TOKEN_EINTRAGEN'
```

Prüfen, ohne den Token auszugeben:

```bash
test -n "$GITHUB_TOKEN" && echo "GITHUB_TOKEN ist gesetzt"
```


## 2. GitHub-Catalog-Modul installieren

Führe den folgenden Befehl im **Hauptverzeichnis deiner Backstage-Installation** aus.

Das GitHub Entity Provider Modul ist nicht standardmässig installiert.


In [ ]:
%%bash
# Diesen Befehl im Root-Verzeichnis der Backstage-Installation ausführen.

yarn --cwd packages/backend add @backstage/plugin-catalog-backend-module-github


## 3. Backend-Modul aktivieren

Öffne diese Datei:

```text
packages/backend/src/index.ts
```

Ergänze das GitHub-Modul direkt bei den Catalog-Backend-Modulen:


In [ ]:
# Inhalt für packages/backend/src/index.ts
#
# Diese Zelle zeigt TypeScript als Text an und verändert keine Datei automatisch.

typescript_code = """backend.add(import('@backstage/plugin-catalog-backend'));
backend.add(import('@backstage/plugin-catalog-backend-module-github'));"""

print(typescript_code)


Der relevante Ausschnitt in `packages/backend/src/index.ts` muss danach mindestens so aussehen:

```typescript
backend.add(import('@backstage/plugin-catalog-backend'));
backend.add(import('@backstage/plugin-catalog-backend-module-github'));
```


## 4. `app-config.yaml` konfigurieren

Ersetze `MEINE-GITHUB-ORGANISATION` durch den exakten Namen deiner GitHub-Organisation.

Die Konfiguration:

- verwendet den Token aus `GITHUB_TOKEN`,
- durchsucht alle Repositories der Organisation,
- sucht im jeweiligen Standard-Branch,
- berücksichtigt nur `/catalog-info.yaml`,
- aktualisiert den Catalog alle 30 Minuten.

Da kein Branch-Filter angegeben ist, verwendet Backstage automatisch den Standard-Branch des jeweiligen Repositorys.


In [ ]:
# Konfiguration für app-config.yaml als kopierbarer Text.

github_organization = "MEINE-GITHUB-ORGANISATION"

config = f"""integrations:
  github:
    - host: github.com
      token: ${{GITHUB_TOKEN}}

catalog:
  providers:
    github:
      meineRepositories:
        organization: '{github_organization}'
        catalogPath: /catalog-info.yaml
        filters:
          repository: .*
        schedule:
          frequency:
            minutes: 30
          timeout:
            minutes: 3
"""

print(config)


Kopierfertige Variante:

```yaml
integrations:
  github:
    - host: github.com
      token: ${GITHUB_TOKEN}

catalog:
  providers:
    github:
      meineRepositories:
        organization: 'MEINE-GITHUB-ORGANISATION'
        catalogPath: /catalog-info.yaml
        filters:
          repository: .*
        schedule:
          frequency:
            minutes: 30
          timeout:
            minutes: 3
```

### Optional: Nur tatsächlich vorhandene Dateien registrieren

Standardmässig ist `validateLocationsExist` deaktiviert. Du kannst es aktivieren, damit Backstage vor dem Erzeugen einer Location prüft, ob die konfigurierte Datei tatsächlich existiert:

```yaml
        validateLocationsExist: true
```

Für den festen Pfad `/catalog-info.yaml` ist diese Option zulässig.


## Ergebnis nach Punkt 4

Nach diesen Schritten sind:

1. der GitHub-Token verfügbar,
2. das GitHub-Catalog-Modul installiert,
3. das Modul im Backend aktiviert,
4. GitHub Discovery in `app-config.yaml` konfiguriert.

Beim Start von Backstage durchsucht der Provider die konfigurierte GitHub-Organisation und importiert die gefundenen `catalog-info.yaml`-Dateien in den Software Catalog.
